In [2]:
import pandas as pd
import numpy as np

# 1. Auction players (master list)
auction = pd.read_excel("Player_List.xlsx", usecols=[0,1,2])
auction.columns = ['Player Name', 'Base Price', 'Category']
auction['Player Name'] = auction['Player Name'].str.strip()
auction.to_csv("auction_players.csv", index=False)

# 2. Past price 2025 ONLY
past = pd.read_csv("auction_summary.csv")
past = past[past['year'] == 2025]
past = past.sort_values('final_price', ascending=False).drop_duplicates('name')
past = past[['name', 'final_price', 'sold_to', 'auction_status']]
past['final_price'] = past['final_price'] / 10_000_000
past.columns = ['Player Name', 'Past_Price_2025', 'Past_Team_2025', 'Sold_2025']
past['Sold_2025'] = past['Sold_2025'].str.upper().str.contains('SOLD').fillna(False)
past = auction[['Player Name']].merge(past, on='Player Name', how='left')
past.to_csv("past_price_2025.csv", index=False)

# 3. Stats → ONE ROW PER PLAYER (2023-2025 only) - FIXED VERSION
stats = pd.read_csv("player_match_stats.csv")
matches = pd.read_csv("matches.csv")
matches['year'] = pd.to_datetime(matches['match_date'], dayfirst=True, errors='coerce').dt.year
recent_stats = stats[stats['match_id'].isin(matches[matches['year'] >= 2023]['match_id'])]

# ====== BOWLING - SAFE WAY ======
bowl_data = recent_stats[recent_stats['overs_bowled'] > 0]

# Total wickets and runs conceded
bowl_totals = bowl_data.groupby('player_name').agg(
    Wickets=('wicket_taken', 'sum'),
    Runs_Conceded=('runs_conceded', 'sum'),
    Bowl_Innings=('match_id', 'nunique')
).reset_index()

# Median economy
eco = bowl_data.groupby('player_name')['economy_rate'].median().round(2).reset_index()
eco.columns = ['player_name', 'Eco_Median']

# Merge and calculate average safely
bowl = bowl_totals.merge(eco, on='player_name')
bowl['Bowl_Avg'] = (bowl['Runs_Conceded'] / bowl['Wickets'].replace(0, 1)).round(2)
bowl = bowl[['player_name', 'Wickets', 'Bowl_Innings', 'Eco_Median', 'Bowl_Avg']]

# ====== BATTING - SAFE WAY ======
bat_data = recent_stats[recent_stats['balls_faced'] > 0]

# Total runs and innings
bat_totals = bat_data.groupby('player_name').agg(
    Runs=('runs_scored', 'sum'),
    Bat_Innings=('match_id', 'nunique')
).reset_index()

# Median strike rate
sr = bat_data.groupby('player_name')['strike_rate'].median().round(2).reset_index()
sr.columns = ['player_name', 'SR_Median']

# Merge and calculate average
bat = bat_totals.merge(sr, on='player_name')
bat['Bat_Avg'] = (bat['Runs'] / bat['Bat_Innings'].replace(0, 1)).round(2)
bat = bat[['player_name', 'Runs', 'Bat_Innings', 'SR_Median', 'Bat_Avg']]

# ====== FINAL ONE ROW PER PLAYER ======
one_row = auction[['Player Name']].merge(bowl, left_on='Player Name', right_on='player_name', how='left')
one_row = one_row.merge(bat, left_on='Player Name', right_on='player_name', how='left')

# Fill missing values
one_row.fillna({
    'Wickets': 0, 'Bowl_Innings': 0, 'Eco_Median': 12.0, 'Bowl_Avg': 99.0,
    'Runs': 0, 'Bat_Innings': 0, 'SR_Median': 100.0, 'Bat_Avg': 0.0
}, inplace=True)

one_row.drop(columns=['player_name_x', 'player_name_y'], errors='ignore', inplace=True)
one_row.to_csv("player_stats_1row.csv", index=False)

# 4. Clean player info
info = pd.read_csv("players.csv")
info = info.drop_duplicates('player_name')
info = info[['player_name','bowling_type','batting_type','batting_hand','date_of_birth','is_wicket_keeper']]
info.columns = ['Player Name','Bowl_Style','Bat_Style','Bat_Hand','DOB','Is_Keeper']
info['Age'] = (pd.to_datetime('2025-11-18') - pd.to_datetime(info['DOB'], errors='coerce')).dt.days // 365
info = auction[['Player Name']].merge(info, on='Player Name', how='left')
info.to_csv("player_info.csv", index=False)

# 5. Current RR squad
rr = pd.DataFrame({
    'Player Name': ['Dhruv Jurel','Donovan Ferreira','Jofra Archer','Kwena Maphaka','Lhuan-Dre Pretorious',
                    'Nandre Burger','Ravindra Jadeja','Riyan Parag','Sam Curran','Sandeep Sharma',
                    'Shimron Hetmyer','Shubham Dubey','Tushar Deshpande','Vaibhav Suryavanshi',
                    'Yashasvi Jaiswal','Yudhvir Charak'],
    'Retained': [True]*16
})
rr.to_csv("rr_current_squad.csv", index=False)

print("FINAL 5 DATASETS CREATED SUCCESSFULLY! NO ERRORS!")
print("→ auction_players.csv")
print("→ past_price_2025.csv") 
print("→ player_stats_1row.csv   ← PERFECT (323 rows, 1 per player)")
print("→ player_info.csv")
print("→ rr_current_squad.csv")
print("\nYou can now delete all huge old files. We're done with data cleaning forever!")

C:\Users\Aksha\AppData\Local\Temp\ipykernel_23160\2142372787.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  matches['year'] = pd.to_datetime(matches['match_date'], dayfirst=True, errors='coerce').dt.year
C:\Users\Aksha\AppData\Local\Temp\ipykernel_23160\2142372787.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  info['Age'] = (pd.to_datetime('2025-11-18') - pd.to_datetime(info['DOB'], errors='coerce')).dt.days // 365


FINAL 5 DATASETS CREATED SUCCESSFULLY! NO ERRORS!
→ auction_players.csv
→ past_price_2025.csv
→ player_stats_1row.csv   ← PERFECT (323 rows, 1 per player)
→ player_info.csv
→ rr_current_squad.csv

You can now delete all huge old files. We're done with data cleaning forever!


In [7]:
import pandas as pd
import os

# Function to standardize names for robust merging
def normalize_name(series):
    return series.astype(str).str.strip().str.upper()

# 1. LOAD THE 5 CLEAN DATASETS
print("Loading datasets...")
try:
    auction_df = pd.read_csv("auction_players.csv")
    info_df = pd.read_csv("player_info.csv")
    stats_df = pd.read_csv("player_stats_1row.csv")
    past_df = pd.read_csv("past_price_2025.csv")
    rr_squad_df = pd.read_csv("rr_current_squad.csv")
except FileNotFoundError as e:
    print(f"Error: Missing file - {e}")
    print("Please ensure you ran the previous cleaning script first!")
    exit()

# 2. CREATE MERGE KEYS (Normalization)
# We use a temporary 'merge_key' to ensure perfect matching regardless of case/spacing
auction_df['merge_key'] = normalize_name(auction_df['Player Name'])
info_df['merge_key'] = normalize_name(info_df['Player Name'])
stats_df['merge_key'] = normalize_name(stats_df['Player Name'])
past_df['merge_key'] = normalize_name(past_df['Player Name'])
rr_squad_df['merge_key'] = normalize_name(rr_squad_df['Player Name'])

# 3. INTELLIGENT MERGING (Left Join on Auction List)
print("Merging datasets into Master File...")

# Start with the Auction List (The Base)
master_df = auction_df.copy()

# Merge 1: Add Player Info (Select columns to avoid duplicates)
# Columns to add: Bowl_Style, Bat_Style, Bat_Hand, Age, Is_Keeper
cols_to_add = ['merge_key', 'Bowl_Style', 'Bat_Style', 'Bat_Hand', 'Age', 'Is_Keeper']
master_df = pd.merge(master_df, info_df[cols_to_add], on='merge_key', how='left')

# Merge 2: Add Stats (Performance)
# Columns to add: Wickets, Bowl_Innings, Eco_Median, Runs, Bat_Innings, SR_Median, Bat_Avg
cols_to_add = ['merge_key', 'Wickets', 'Bowl_Innings', 'Eco_Median', 'Bowl_Avg', 'Runs', 'Bat_Innings', 'SR_Median', 'Bat_Avg']
master_df = pd.merge(master_df, stats_df[cols_to_add], on='merge_key', how='left')

# Merge 3: Add Past Price Info (Context)
# Columns to add: Past_Price_2025, Past_Team_2025, Sold_2025
cols_to_add = ['merge_key', 'Past_Price_2025', 'Past_Team_2025', 'Sold_2025']
master_df = pd.merge(master_df, past_df[cols_to_add], on='merge_key', how='left')

# Merge 4: Add RR Squad Status (To filter them out later)
master_df = pd.merge(master_df, rr_squad_df[['merge_key', 'Retained']], on='merge_key', how='left')

# 4. FINAL CLEANUP
# Fill missing numerical stats with 0 (uncapped/no data)
stat_cols = ['Wickets', 'Bowl_Innings', 'Eco_Median', 'Runs', 'Bat_Innings', 'SR_Median', 'Bat_Avg', 'Past_Price_2025', 'Age']
master_df[stat_cols] = master_df[stat_cols].fillna(0)

# Fill text/boolean columns
master_df['Retained'] = master_df['Retained'].fillna(False)
master_df['Sold_2025'] = master_df['Sold_2025'].fillna(False)
master_df['Past_Team_2025'] = master_df['Past_Team_2025'].fillna('None')

# Drop the temporary merge key
master_df = master_df.drop(columns=['merge_key'])



# Save
output_filename = "Final_Auction_Master.csv"
master_df.to_csv(output_filename, index=False)

print(f"\nSUCCESS! All files merged into '{output_filename}'")
print(f"Total Players: {len(master_df)}")
print("Columns:", list(master_df.columns))
print("\nSample Data:")
print(master_df[['Player Name', 'Base Price', 'SR_Median', 'Wickets']].head())

Loading datasets...
Merging datasets into Master File...

SUCCESS! All files merged into 'Final_Auction_Master.csv'
Total Players: 350
Columns: ['Player Name', 'Base Price', 'Category', 'Bowl_Style', 'Bat_Style', 'Bat_Hand', 'Age', 'Is_Keeper', 'Wickets', 'Bowl_Innings', 'Eco_Median', 'Bowl_Avg', 'Runs', 'Bat_Innings', 'SR_Median', 'Bat_Avg', 'Past_Price_2025', 'Past_Team_2025', 'Sold_2025', 'Retained']

Sample Data:
             Player Name  Base Price  SR_Median  Wickets
0  Daryl Joseph Mitchell         1.0     130.77     20.0
1             Tom Banton         1.0     133.33      0.0
2           Jason Holder         1.0     127.27    131.0
3             Finn Allen         1.0     156.60      0.0
4            Jake McGurk         1.0     128.57      0.0


C:\Users\Aksha\AppData\Local\Temp\ipykernel_23160\3884598066.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_df['Retained'] = master_df['Retained'].fillna(False)
C:\Users\Aksha\AppData\Local\Temp\ipykernel_23160\3884598066.py:60: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_df['Sold_2025'] = master_df['Sold_2025'].fillna(False)
